<a href="https://colab.research.google.com/github/minaGalil/Imperial-Capstone/blob/main/Capstone_Week_3_Calling_Functions_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# =========================================================
# SETTINGS
# =========================================================
np.random.seed(42)

# Week 1 submitted points
week1_points = {
    1: [0.761024, 0.713000],
    2: [0.732637, 0.906564],
    3: [0.522581, 0.591593, 0.350176],
    4: [0.564020, 0.473834, 0.390972, 0.258427],
    5: [0.196777, 0.892275, 0.855813, 0.891829],
    6: [0.728785, 0.146949, 0.767044, 0.739699, 0.020789],
    7: [0.016700, 0.532618, 0.280647, 0.222769, 0.407487, 0.748018],
    8: [0.035844, 0.064098, 0.010711, 0.024463, 0.361511, 0.783344, 0.515157, 0.880442],
}

# Week 1 returned outputs
week1_outputs = {
    1: -7.565331332744927e-18,
    2: 0.5530064492925906,
    3: -0.03333218343962805,
    4: -4.83054096204485,
    5: 1257.680268889983,
    6: -0.8072367077314392,
    7: 1.2394822144938658,
    8: 9.5430069620611,
}

# Week 2 submitted points
week2_points = {
    1: [0.751825, 0.811946],
    2: [0.980817, 0.626715],
    3: [0.996446, 0.737055, 0.850924],
    4: [0.404477, 0.413254, 0.303108, 0.434359],
    5: [0.521278, 0.505138, 0.985473, 0.994545],
    6: [0.341155, 0.021278, 0.626487, 0.970661, 0.032762],
    7: [0.431390, 0.173879, 0.071339, 0.124473, 0.403592, 0.624180],
    8: [0.064214, 0.412793, 0.081589, 0.008195, 0.974438, 0.216196, 0.139173, 0.110624],
}

# Week 2 returned outputs
week2_outputs = {
    1: 2.495129202697582e-35,
    2: 0.06646691679004207,
    3: -0.04126707799435369,
    4: -0.7158150747637886,
    5: 1769.2992166742467,
    6: -0.721361811601727,
    7: 1.493591747104962,
    8: 9.7741312776374,
}

all_results = []

# =========================================================
# LOOP OVER ALL FUNCTIONS
# =========================================================
for func_id in range(1, 9):

    print("\n" + "=" * 60)
    print(f"Function {func_id} - Week 3")
    print("=" * 60)

    # -----------------------------------------------------
    # LOAD ORIGINAL DATA
    # -----------------------------------------------------
    X = np.load(f"function{func_id}_inputs.npy")
    y = np.load(f"function{func_id}_outputs.npy")

    # -----------------------------------------------------
    # APPEND WEEK 1 AND WEEK 2 DATA
    # -----------------------------------------------------
    for point, output in [
        (week1_points[func_id], week1_outputs[func_id]),
        (week2_points[func_id], week2_outputs[func_id]),
    ]:
        point = np.array(point).flatten()

        if len(point) != X.shape[1]:
            raise ValueError(
                f"Function {func_id}: point dimension {len(point)} != expected {X.shape[1]}"
            )

        exists = np.any(np.all(np.isclose(X, point.reshape(1, -1)), axis=1))
        if not exists:
            X = np.vstack([X, point])
            y = np.append(y, output)

    print("Dataset shape after Week 1 + Week 2 append:", X.shape)

    # -----------------------------------------------------
    # CURRENT BEST
    # -----------------------------------------------------
    best_idx = np.argmax(y)
    best_input = X[best_idx]
    best_output = y[best_idx]

    print("Current best output:", best_output)
    print("Current best input:", best_input)

    dim = X.shape[1]

    # -----------------------------------------------------
    # DIMENSION-BASED SETTINGS
    # -----------------------------------------------------
    if dim <= 3:
        beta = 1.5
        n_candidates = 6000
        noise_scale = 0.03
    elif dim <= 5:
        beta = 2.0
        n_candidates = 9000
        noise_scale = 0.05
    else:
        beta = 2.5
        n_candidates = 14000
        noise_scale = 0.08

    # -----------------------------------------------------
    # GP SURROGATE MODEL
    # -----------------------------------------------------
    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * RBF(length_scale=np.ones(dim), length_scale_bounds=(1e-2, 1e2))
        + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=5,
        random_state=42,
    )

    gp.fit(X, y)

    # -----------------------------------------------------
    # SVM CLASSIFIER: PROMISING REGION VS OTHERS
    # Label top 30% outputs as promising
    # -----------------------------------------------------
    threshold = np.quantile(y, 0.70)
    labels = (y >= threshold).astype(int)

    use_svm = len(np.unique(labels)) > 1

    if use_svm:
        svm_model = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf", probability=True, C=1.0, gamma="scale", random_state=42))
        ])
        svm_model.fit(X, labels)
    else:
        svm_model = None

    # -----------------------------------------------------
    # CANDIDATES
    # -----------------------------------------------------
    candidates_global = np.random.uniform(0, 1, size=(n_candidates, dim))
    local_noise = np.random.normal(0, noise_scale, size=(1500, dim))
    candidates_local = np.clip(best_input + local_noise, 0, 1)

    candidates = np.vstack([candidates_global, candidates_local])

    # -----------------------------------------------------
    # GP ACQUISITION: UCB
    # -----------------------------------------------------
    mu, sigma = gp.predict(candidates, return_std=True)
    ucb = mu + beta * sigma

    # -----------------------------------------------------
    # SVM GUIDANCE
    # -----------------------------------------------------
    if use_svm:
        svm_prob = svm_model.predict_proba(candidates)[:, 1]
    else:
        svm_prob = np.ones(len(candidates))

    # -----------------------------------------------------
    # HYBRID SCORE
    # -----------------------------------------------------
    hybrid_score = ucb * (0.5 + 0.5 * svm_prob)

    best_candidate_idx = np.argmax(hybrid_score)
    new_point = candidates[best_candidate_idx]
    new_point = np.clip(new_point, 0, 1)

    predicted_mean = mu[best_candidate_idx]
    predicted_std = sigma[best_candidate_idx]
    predicted_svm_prob = svm_prob[best_candidate_idx]

    query_string = "-".join([f"{v:.6f}" for v in new_point])

    print("New point:", new_point)
    print("Predicted mean:", predicted_mean)
    print("Predicted std:", predicted_std)
    print("SVM promising probability:", predicted_svm_prob)
    print("Week 3 FINAL SUBMISSION:", query_string)

    all_results.append({
        "Function": func_id,
        "Dimension": dim,
        "Current_Best_Output": float(best_output),
        "Beta": beta,
        "Threshold_for_SVM": float(threshold),
        "Predicted_Mean": float(predicted_mean),
        "Predicted_Std": float(predicted_std),
        "Predicted_SVM_Prob": float(predicted_svm_prob),
        "Week3_Submission": query_string
    })

# =========================================================
# SAVE RESULTS
# =========================================================
df = pd.DataFrame(all_results)
df.to_csv("Week3_Submissions.csv", index=False)

print("\nAll Week 3 submissions saved to Week3_Submissions.csv")


Function 1 - Week 3
Dataset shape after Week 1 + Week 2 append: (12, 2)
Current best output: 7.710875114502849e-16
Current best input: [0.73102363 0.73299988]


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


New point: [0.70262958 0.95598006]
Predicted mean: 0.0005252573774032472
Predicted std: 0.0005790728983513588
SVM promising probability: 0.396492768423633
Week 3 FINAL SUBMISSION: 0.702630-0.955980

Function 2 - Week 3
Dataset shape after Week 1 + Week 2 append: (12, 2)
Current best output: 0.6112052157614438
Current best input: [0.70263656 0.9265642 ]


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


New point: [0.70032878 1.        ]
Predicted mean: 0.6127896182315351
Predicted std: 0.035714057208059286
SVM promising probability: 0.3662520373905922
Week 3 FINAL SUBMISSION: 0.700329-1.000000

Function 3 - Week 3
Dataset shape after Week 1 + Week 2 append: (17, 3)
Current best output: -0.03333218343962805
Current best input: [0.522581 0.591593 0.350176]


...

New point: [0.44932454 0.60371538 1.         1.        ]
Predicted mean: 1807.343675179905
Predicted std: 75.27401118879227
SVM promising probability: 0.8866587650642376
Week 3 FINAL SUBMISSION: 0.449325-0.603715-1.000000-1.000000

Function 6 - Week 3
Dataset shape after Week 1 + Week 2 append: (22, 5)
Current best output: -0.7142649478202404
Current best input: [0.7281861  0.15469257 0.73255167 0.69399651 0.05640131]


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


New point: [0.94115509 0.65410082 0.07924822 0.96113727 0.22974582]
Predicted mean: -0.6983121760878972
Predicted std: 0.3113539402111557
SVM promising probability: 0.2582016870163739
Week 3 FINAL SUBMISSION: 0.941155-0.654101-0.079248-0.961137-0.229746

Function 7 - Week 3
Dataset shape after Week 1 + Week 2 append: (32, 6)
Current best output: 1.493591747104962
Current best input: [0.43139  0.173879 0.071339 0.124473 0.403592 0.62418 ]


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


New point: [0.1531004  0.23673666 0.19001802 0.         0.36512752 0.74415508]
Predicted mean: 1.19877633469308
Predicted std: 0.1983539366125058
SVM promising probability: 0.7025097222070372
Week 3 FINAL SUBMISSION: 0.153100-0.236737-0.190018-0.000000-0.365128-0.744155

Function 8 - Week 3
Dataset shape after Week 1 + Week 2 append: (42, 8)
Current best output: 9.7741312776374
Current best input: [0.064214 0.412793 0.081589 0.008195 0.974438 0.216196 0.139173 0.110624]
New point: [0.10376189 0.01760615 0.20475138 0.19410134 0.74536723 0.67418158
 0.12536188 0.04402382]
Predicted mean: 10.04332087136676
Predicted std: 0.13251170649586558
SVM promising probability: 0.9819751318524031
Week 3 FINAL SUBMISSION: 0.103762-0.017606-0.204751-0.194101-0.745367-0.674182-0.125362-0.044024

All Week 3 submissions saved to Week3_Submissions.csv
